# 05 — Ontology-informed machine learning (extreme-group setting)

This notebook compares conventional and ontology-informed predictive representations for short-term motor progression.

**Prediction target**:
- `progression_group`
- retained classes: `slow`, `faster`
- excluded class: `intermediate`

**Inputs**:
- `output/trajectory_analysis/subject_progression_table.csv`
- `output/trajectory_analysis/progression_visit_level_table.csv`
- `mapping/ppmi_pdon_pmdo_mapping.csv`

**Representations evaluated**:
1. **Conventional / all predictors**: baseline raw variables used directly as model inputs.
2. **Conventional / target-excluded**: same as above, but excluding `updrs3_score`, which is the variable used to define the progression label.
3. **Ontology-informed / all predictors**: baseline variables reorganised into semantic domain features and concept-level aggregates.
4. **Ontology-informed / target-excluded**: ontology-informed representation with target-derived motor contribution recomputed without `updrs3_score`.

**Models**:
- Logistic regression
- Support vector machine (RBF)
- Decision tree
- Random forest
- k-nearest neighbours
- Gaussian naive Bayes
- Gradient boosting

**Main outputs**:
- baseline modelling dataset
- feature matrices
- cross-validated metrics
- confusion matrices
- feature summaries saved to `output/ontology_informed_ml/`


In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn

## 1) Project root (Drive recommended)

In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ppmi-ontology-alignment')
else:
    PROJECT_DIR = Path('/content/ppmi-ontology-alignment')

MAP_DIR = PROJECT_DIR / 'mapping'
OUT_DIR = PROJECT_DIR / 'output'
TRAJ_DIR = OUT_DIR / 'trajectory_analysis'
ML_DIR = OUT_DIR / 'ontology_informed_ml'
FIG_DIR = ML_DIR / 'figures'
for p in [MAP_DIR, OUT_DIR, TRAJ_DIR, ML_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MAP_PATH = MAP_DIR / 'ppmi_pdon_pmdo_mapping.csv'
PROG_PATH = TRAJ_DIR / 'subject_progression_table.csv'
VISIT_PATH = TRAJ_DIR / 'progression_visit_level_table.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('MAP_PATH   :', MAP_PATH, 'exists=', MAP_PATH.exists())
print('PROG_PATH  :', PROG_PATH, 'exists=', PROG_PATH.exists())
print('VISIT_PATH :', VISIT_PATH, 'exists=', VISIT_PATH.exists())
print('ML_DIR     :', ML_DIR)

## 2) Load inputs

In [ ]:
import pandas as pd
import numpy as np

if not MAP_PATH.exists():
    raise FileNotFoundError(f'Missing mapping file: {MAP_PATH}')
if not PROG_PATH.exists():
    raise FileNotFoundError(f'Missing progression file: {PROG_PATH}')
if not VISIT_PATH.exists():
    raise FileNotFoundError(f'Missing visit-level file: {VISIT_PATH}')

mapping = pd.read_csv(MAP_PATH, dtype=str).fillna('')
progression = pd.read_csv(PROG_PATH)
visit_level = pd.read_csv(VISIT_PATH)

print('Mapping rows:', len(mapping))
print('Progression rows:', len(progression))
print('Visit-level rows:', len(visit_level))
print('Unique subjects in progression:', progression['PATNO'].astype(str).nunique())
print('Unique subjects in visit-level:', visit_level['PATNO'].astype(str).nunique())
progression.head(3)

## 3) Define baseline modelling dataset

This notebook uses the first visit of the fixed two-visit pair (`V04`) as the predictor time point and retains only the extreme progression groups (`slow`, `faster`) for prediction.

In [ ]:
def normalize_event_id(x):
    return str(x).strip().upper()

visit_level['PATNO'] = visit_level['PATNO'].astype(str).str.strip()
visit_level['EVENT_ID_norm'] = (
    visit_level['EVENT_ID_norm'].astype(str).str.strip().str.upper()
    if 'EVENT_ID_norm' in visit_level.columns
    else visit_level['EVENT_ID'].apply(normalize_event_id)
)

progression['PATNO'] = progression['PATNO'].astype(str).str.strip()
progression['progression_group'] = progression['progression_group'].astype(str).str.strip().str.lower()

baseline = visit_level[visit_level['EVENT_ID_norm'] == 'V04'].copy()

for col in ['progression_group', 'annualised_updrs3_change',
            'progression_group_x', 'progression_group_y',
            'annualised_updrs3_change_x', 'annualised_updrs3_change_y']:
    if col in baseline.columns:
        baseline = baseline.drop(columns=[col])

baseline = baseline.merge(
    progression[['PATNO', 'progression_group', 'annualised_updrs3_change']],
    on='PATNO',
    how='inner'
)

baseline = baseline[baseline['progression_group'].isin(['slow', 'faster'])].copy()
baseline['progression_label'] = baseline['progression_group'].map({'slow': 0, 'faster': 1})

print('Baseline modelling rows:', len(baseline))
print('Unique subjects:', baseline['PATNO'].nunique())
print('Class distribution:')
print(baseline['progression_group'].value_counts())
baseline.head(3)

## 4) Define raw variables and semantic domains

In [ ]:
SUBJECT_META = ['SEX', 'EDUCYRS', 'fampd_bin', 'APOE']
VISIT_META = ['age_at_visit']
MOTOR_VARS = ['updrs1_score', 'updrs2_score', 'updrs3_score', 'updrs4_score', 'hy']
COGNITION_VARS = ['moca', 'cogstate', 'MCI_testscores']
BIOMARKER_VARS = ['abeta', 'ptau', 'asyn']
IMAGING_VARS = [
    'MIA_LOWPUT_EXPECTED',
    'MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT',
    'MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT',
    'MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'
]
TARGET_VAR = 'updrs3_score'

RAW_FEATURES_ALL = SUBJECT_META + VISIT_META + MOTOR_VARS + COGNITION_VARS + BIOMARKER_VARS + IMAGING_VARS
RAW_FEATURES_ALL = [c for c in RAW_FEATURES_ALL if c in baseline.columns]
RAW_FEATURES_NO_TARGET = [c for c in RAW_FEATURES_ALL if c != TARGET_VAR]

print('Raw features (all):', RAW_FEATURES_ALL)
print('N raw features (all):', len(RAW_FEATURES_ALL))
print('\nRaw features (target-excluded):', RAW_FEATURES_NO_TARGET)
print('N raw features (target-excluded):', len(RAW_FEATURES_NO_TARGET))

## 5) Build conventional feature matrices

In [ ]:
def to_float_safe(x):
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def build_conventional_matrix(df, feature_list):
    X = df[['PATNO'] + feature_list].copy()
    for c in feature_list:
        X[c] = X[c].apply(to_float_safe)
    if 'APOE' in df.columns and 'APOE' in X.columns:
        apoe_raw = df['APOE'].astype(str).str.lower().fillna('')
        X['APOE_e4_present'] = apoe_raw.apply(lambda s: 1.0 if 'e4' in s else 0.0)
        X = X.drop(columns=['APOE'])
    feature_cols = [c for c in X.columns if c != 'PATNO']
    return X, feature_cols

X_raw_all, raw_all_feature_cols = build_conventional_matrix(baseline, RAW_FEATURES_ALL)
X_raw_no_target, raw_no_target_feature_cols = build_conventional_matrix(baseline, RAW_FEATURES_NO_TARGET)
y = baseline['progression_label'].copy()

print('X_raw_all shape      :', X_raw_all[raw_all_feature_cols].shape)
print('X_raw_no_target shape:', X_raw_no_target[raw_no_target_feature_cols].shape)
print('y shape              :', y.shape)
X_raw_all.head(3)

## 6) Build ontology-informed feature matrices

The ontology-informed representation reorganises baseline variables into semantic domain and concept-level aggregates.

Two versions are created:
- **all predictors**
- **target-excluded**, where `updrs3_score` is removed from the direct motor contribution.

In [ ]:
numeric_baseline = baseline.copy()
for c in RAW_FEATURES_ALL:
    if c in numeric_baseline.columns:
        numeric_baseline[c] = numeric_baseline[c].apply(to_float_safe)


def build_ontology_matrix(df, exclude_target=False):
    X = pd.DataFrame({'PATNO': df['PATNO'].astype(str).values})

    if 'SEX' in df.columns:
        X['sex_numeric'] = df['SEX'].apply(to_float_safe)
    if 'EDUCYRS' in df.columns:
        X['education_years'] = df['EDUCYRS'].apply(to_float_safe)
    if 'fampd_bin' in df.columns:
        X['family_history_pd'] = df['fampd_bin'].apply(to_float_safe)
    if 'age_at_visit' in df.columns:
        X['age_at_visit'] = df['age_at_visit'].apply(to_float_safe)
    if 'APOE' in df.columns:
        apoe_raw = df['APOE'].astype(str).str.lower().fillna('')
        X['APOE_e4_present'] = apoe_raw.apply(lambda s: 1.0 if 'e4' in s else 0.0)

    motor_cols = [c for c in MOTOR_VARS if c in df.columns]
    if exclude_target:
        motor_cols = [c for c in motor_cols if c != TARGET_VAR]

    cognition_cols = [c for c in COGNITION_VARS if c in df.columns]
    biomarker_cols = [c for c in BIOMARKER_VARS if c in df.columns]
    imaging_cols = [c for c in IMAGING_VARS if c in df.columns]

    if motor_cols:
        X['motor_mean'] = df[motor_cols].mean(axis=1, skipna=True)
        X['motor_sum'] = df[motor_cols].sum(axis=1, skipna=True)
    if cognition_cols:
        X['cognition_mean'] = df[cognition_cols].mean(axis=1, skipna=True)
    if biomarker_cols:
        X['biomarker_mean'] = df[biomarker_cols].mean(axis=1, skipna=True)
    if imaging_cols:
        X['imaging_mean'] = df[imaging_cols].mean(axis=1, skipna=True)

    caudate_cols = [c for c in ['MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT'] if c in df.columns]
    putamen_cols = [c for c in ['MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT'] if c in df.columns]
    striatum_cols = [c for c in ['MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'] if c in df.columns]

    if caudate_cols:
        X['caudate_mean'] = df[caudate_cols].mean(axis=1, skipna=True)
    if putamen_cols:
        X['putamen_mean'] = df[putamen_cols].mean(axis=1, skipna=True)
    if striatum_cols:
        X['striatum_mean'] = df[striatum_cols].mean(axis=1, skipna=True)

    X['motor_n_obs'] = df[motor_cols].notna().sum(axis=1) if motor_cols else 0
    X['cognition_n_obs'] = df[cognition_cols].notna().sum(axis=1) if cognition_cols else 0
    X['biomarker_n_obs'] = df[biomarker_cols].notna().sum(axis=1) if biomarker_cols else 0
    X['imaging_n_obs'] = df[imaging_cols].notna().sum(axis=1) if imaging_cols else 0

    feature_cols = [c for c in X.columns if c != 'PATNO']
    return X, feature_cols

X_onto_all, onto_all_feature_cols = build_ontology_matrix(numeric_baseline, exclude_target=False)
X_onto_no_target, onto_no_target_feature_cols = build_ontology_matrix(numeric_baseline, exclude_target=True)

print('Ontology-informed features (all):', onto_all_feature_cols)
print('N ontology-informed features (all):', len(onto_all_feature_cols))
print('\nOntology-informed features (target-excluded):', onto_no_target_feature_cols)
print('N ontology-informed features (target-excluded):', len(onto_no_target_feature_cols))
X_onto_all.head(3)

## 7) Prepare models and cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

class_counts = y.value_counts()
min_class_size = int(class_counts.min())
n_splits = min(5, min_class_size)
if n_splits < 2:
    raise ValueError('Not enough samples per class for stratified cross-validation.')

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall'
}

models = {
    'logreg': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'svm_rbf': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=False, random_state=42))
    ]),
    'decision_tree': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, random_state=42))
    ]),
    'rf': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=300, random_state=42))
    ]),
    'knn': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'gaussian_nb': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', GaussianNB())
    ]),
    'gradient_boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', GradientBoostingClassifier(random_state=42))
    ])
}

print('Cross-validation splits:', n_splits)
print('Class counts:')
print(class_counts)
print('Models:', list(models.keys()))

## 8) Evaluate conventional and ontology-informed representations

In [ ]:
results = []
predictions = {}

feature_sets = {
    'conventional_all': (X_raw_all[raw_all_feature_cols], raw_all_feature_cols),
    'conventional_no_target': (X_raw_no_target[raw_no_target_feature_cols], raw_no_target_feature_cols),
    'ontology_informed_all': (X_onto_all[onto_all_feature_cols], onto_all_feature_cols),
    'ontology_informed_no_target': (X_onto_no_target[onto_no_target_feature_cols], onto_no_target_feature_cols)
}

for rep_name, (Xmat, feature_names) in feature_sets.items():
    for model_name, model in models.items():
        cv_res = cross_validate(model, Xmat, y, cv=cv, scoring=scoring, return_train_score=False)
        y_pred = cross_val_predict(model, Xmat, y, cv=cv)
        predictions[(rep_name, model_name)] = y_pred
        results.append({
            'representation': rep_name,
            'model': model_name,
            'n_features': len(feature_names),
            'cv_accuracy_mean': float(np.mean(cv_res['test_accuracy'])),
            'cv_accuracy_std': float(np.std(cv_res['test_accuracy'])),
            'cv_balanced_accuracy_mean': float(np.mean(cv_res['test_balanced_accuracy'])),
            'cv_balanced_accuracy_std': float(np.std(cv_res['test_balanced_accuracy'])),
            'cv_f1_mean': float(np.mean(cv_res['test_f1'])),
            'cv_f1_std': float(np.std(cv_res['test_f1'])),
            'cv_precision_mean': float(np.mean(cv_res['test_precision'])),
            'cv_precision_std': float(np.std(cv_res['test_precision'])),
            'cv_recall_mean': float(np.mean(cv_res['test_recall'])),
            'cv_recall_std': float(np.std(cv_res['test_recall']))
        })

results_df = pd.DataFrame(results).sort_values(['model', 'representation']).reset_index(drop=True)
results_df

## 9) Confusion matrices

In [ ]:
import matplotlib.pyplot as plt

labels = [0, 1]
label_names = ['slow', 'faster']
cm_rows = []

for (rep_name, model_name), y_pred in predictions.items():
    cm = confusion_matrix(y, y_pred, labels=labels)
    cm_rows.append({
        'representation': rep_name,
        'model': model_name,
        'confusion_matrix': cm
    })

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.xticks(range(len(label_names)), label_names)
    plt.yticks(range(len(label_names)), label_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion matrix: {rep_name} / {model_name}')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
    plt.tight_layout()
    out_path = FIG_DIR / f'confusion_matrix_{rep_name}_{model_name}.png'
    plt.savefig(out_path, dpi=200)
    plt.show()

print('Saved confusion matrices to:', FIG_DIR)

## 10) Simple feature inspection

This section reports feature names for all representations and fits tree-based and linear models once on the full dataset for exploratory inspection only. These fitted importances or coefficients are descriptive and should not be interpreted as out-of-sample validation results.

In [ ]:
feature_catalog = pd.DataFrame(
    [{'representation': 'conventional_all', 'feature': f} for f in raw_all_feature_cols] +
    [{'representation': 'conventional_no_target', 'feature': f} for f in raw_no_target_feature_cols] +
    [{'representation': 'ontology_informed_all', 'feature': f} for f in onto_all_feature_cols] +
    [{'representation': 'ontology_informed_no_target', 'feature': f} for f in onto_no_target_feature_cols]
)
feature_catalog.head(20)

In [ ]:
exploratory_outputs = []

for rep_name, (Xmat, feature_names) in feature_sets.items():
    # Logistic regression coefficients
    lr_model = models['logreg']
    lr_model.fit(Xmat, y)
    lr_clf = lr_model.named_steps['clf']
    lr_coef = np.abs(lr_clf.coef_).reshape(-1)
    lr_df = pd.DataFrame({
        'representation': rep_name,
        'model': 'logreg',
        'feature': feature_names,
        'importance': lr_coef
    }).sort_values('importance', ascending=False)
    exploratory_outputs.append(lr_df)

    # Decision tree importances
    dt_model = models['decision_tree']
    dt_model.fit(Xmat, y)
    dt_clf = dt_model.named_steps['clf']
    dt_df = pd.DataFrame({
        'representation': rep_name,
        'model': 'decision_tree',
        'feature': feature_names,
        'importance': dt_clf.feature_importances_
    }).sort_values('importance', ascending=False)
    exploratory_outputs.append(dt_df)

    # Random forest importances
    rf_model = models['rf']
    rf_model.fit(Xmat, y)
    rf_clf = rf_model.named_steps['clf']
    rf_df = pd.DataFrame({
        'representation': rep_name,
        'model': 'rf',
        'feature': feature_names,
        'importance': rf_clf.feature_importances_
    }).sort_values('importance', ascending=False)
    exploratory_outputs.append(rf_df)

    # Gradient boosting importances
    gb_model = models['gradient_boosting']
    gb_model.fit(Xmat, y)
    gb_clf = gb_model.named_steps['clf']
    gb_df = pd.DataFrame({
        'representation': rep_name,
        'model': 'gradient_boosting',
        'feature': feature_names,
        'importance': gb_clf.feature_importances_
    }).sort_values('importance', ascending=False)
    exploratory_outputs.append(gb_df)

feature_importance_df = pd.concat(exploratory_outputs, ignore_index=True)
feature_importance_df.head(20)

## 11) Export results

In [ ]:
baseline.to_csv(ML_DIR / 'baseline_modelling_dataset_extreme_groups.csv', index=False)
X_raw_all.to_csv(ML_DIR / 'conventional_feature_matrix_all_extreme_groups.csv', index=False)
X_raw_no_target.to_csv(ML_DIR / 'conventional_feature_matrix_no_target_extreme_groups.csv', index=False)
X_onto_all.to_csv(ML_DIR / 'ontology_informed_feature_matrix_all_extreme_groups.csv', index=False)
X_onto_no_target.to_csv(ML_DIR / 'ontology_informed_feature_matrix_no_target_extreme_groups.csv', index=False)
results_df.to_csv(ML_DIR / 'cross_validated_metrics_extreme_groups.csv', index=False)
feature_catalog.to_csv(ML_DIR / 'feature_catalog_extreme_groups.csv', index=False)
feature_importance_df.to_csv(ML_DIR / 'exploratory_feature_importances_extreme_groups.csv', index=False)

cm_export_rows = []
for row in cm_rows:
    cm = row['confusion_matrix']
    for i, true_lab in enumerate(label_names):
        for j, pred_lab in enumerate(label_names):
            cm_export_rows.append({
                'representation': row['representation'],
                'model': row['model'],
                'true_label': true_lab,
                'predicted_label': pred_lab,
                'count': int(cm[i, j])
            })
confusion_export = pd.DataFrame(cm_export_rows)
confusion_export.to_csv(ML_DIR / 'confusion_matrices_extreme_groups.csv', index=False)

print('Wrote:')
for fn in [
    'baseline_modelling_dataset_extreme_groups.csv',
    'conventional_feature_matrix_all_extreme_groups.csv',
    'conventional_feature_matrix_no_target_extreme_groups.csv',
    'ontology_informed_feature_matrix_all_extreme_groups.csv',
    'ontology_informed_feature_matrix_no_target_extreme_groups.csv',
    'cross_validated_metrics_extreme_groups.csv',
    'feature_catalog_extreme_groups.csv',
    'exploratory_feature_importances_extreme_groups.csv',
    'confusion_matrices_extreme_groups.csv'
]:
    print('-', ML_DIR / fn)

## 12) Quick interpretation-ready outputs

In [ ]:
print('--- cross_validated_metrics_extreme_groups ---')
print(results_df.round(3).to_string(index=False))

print('\n--- top exploratory feature importances / coefficients ---')
for rep in ['conventional_all', 'conventional_no_target', 'ontology_informed_all', 'ontology_informed_no_target']:
    for mdl in ['logreg', 'decision_tree', 'rf', 'gradient_boosting']:
        print(f'\n[{rep} / {mdl}]')
        sub = feature_importance_df[(feature_importance_df['representation'] == rep) & (feature_importance_df['model'] == mdl)].head(10)
        print(sub[['feature', 'importance']].round(4).to_string(index=False))